# LLM Assistant Training Notebook
This notebook is designed for Google Colab training.

## Setup Instructions
1. Upload this notebook to Google Colab
2. Run cells sequentially
3. Monitor training with W&B dashboard

## 1. Install Dependencies

In [ ]:
# Clone repository
!git clone https://github.com/aipulse54-svg/Josh_bot.git
%cd Josh_bot

# Install dependencies
!pip install -q -r requirements.txt

## 2. Setup Weights & Biases Logging

In [ ]:
import wandb

# Login to W&B
wandb.login()

## 3. Load Configuration and Data

In [ ]:
import yaml
import logging
from src.data.dataset import CodeMathDataset
from transformers import AutoTokenizer

# Setup logging
logging.basicConfig(level=logging.INFO)

# Load config
with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print('Configuration loaded')
print(f"Model: {config['model']['name']}")

## 4. Load and Prepare Datasets

In [ ]:
# Initialize tokenizer
model_name = config['model']['name']
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Prepare datasets
dataset_handler = CodeMathDataset(
    tokenizer=tokenizer,
    max_seq_length=config['dataset']['max_seq_length'],
    num_workers=config['dataset']['preprocessing_num_workers'],
)

train_dataset, val_dataset = dataset_handler.prepare_datasets()
print(f'Train dataset size: {len(train_dataset)}')
print(f'Val dataset size: {len(val_dataset)}')

## 5. Initialize Model

In [ ]:
import torch
from src.model.llm import LLMAssistant

# Initialize model with LoRA
llm_model = LLMAssistant(
    model_name=model_name,
    use_lora=config['lora']['enabled'],
    lora_config=config['lora'],
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

print('Model initialized successfully')

## 6. Start Training

In [ ]:
from src.training.trainer import LLMTrainer

# Initialize trainer
trainer = LLMTrainer(
    model=llm_model.get_model(),
    tokenizer=llm_model.get_tokenizer(),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    config=config,
)

# Train
metrics = trainer.train()
print('Training completed!')
print(f'Final metrics: {metrics}')

## 7. Test Inference

In [ ]:
from src.inference.assistant import CodeMathAssistant

# Initialize assistant
assistant = CodeMathAssistant(llm_model)

# Test code generation
result = assistant.generate_code('Write a function to calculate fibonacci numbers')
print('Generated Code:')
print(result['code'])

## 8. Save Model

In [ ]:
# Save model to drive
from google.colab import drive

drive.mount('/content/drive')

import shutil
shutil.copytree('outputs/checkpoint', '/content/drive/MyDrive/llm_assistant_checkpoint')